In [ ]:
# Cell 1: Load GitHub PAT from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
pat = secrets.get_secret('GITHUB_PAT')
os.environ['GITHUB_PAT'] = pat
print('PAT loaded OK')

In [ ]:
# Cell 2: Clone repo (branch dengan fixes)
import os
pat = os.environ['GITHUB_PAT']

# cd to safe dir first to avoid getcwd error when rm -rf deletes cwd
%cd /kaggle/working
!rm -rf EMA-SKD
!git clone https://{pat}@github.com/almaas-izdihar/ema-skd EMA-SKD
%cd EMA-SKD
!git checkout experiment/ablation-alignment
!git log --oneline -5

In [ ]:
# Cell 3: Verify GPU
!nvidia-smi

In [ ]:
# Cell 3.5: Run Config — set USE_DDP and epochs here
USE_DDP   = True    # True = both T4 GPUs via mp.spawn; False = single GPU
END_EPOCH = 200     # 5 for smoke test, 200 for full run
BATCH     = 256     # total batch size (128 per GPU × 2 with DDP)
WORKERS   = 8       # total dataloader workers (4 per GPU × 2 with DDP)

import torch
_ngpu     = torch.cuda.device_count()
_ddp_flag = '--multiprocessing_distributed' if USE_DDP else ''
_bs       = BATCH     if USE_DDP else BATCH   // 2
_wk       = WORKERS   if USE_DDP else WORKERS // 2
print(f'DDP={USE_DDP}  GPUs={_ngpu}  batch_total={BATCH}  batch_per_gpu={_bs}  workers={_wk}  epochs={END_EPOCH}')


In [ ]:
# Cell 4: Run 1 — Baseline (L_CE only, no EHSKD)
!python main.py \
  --data_type cifar100 \
  --data_path /kaggle/working/data \
  --classifier_type ResNet18 \
  --batch_size {_bs} \
  --end_epoch {END_EPOCH} \
  --workers {_wk} \
  --seed 2024 \
  {_ddp_flag} \
  --experiment_type s0_baseline


In [ ]:
# Cell 5: Run 2 — EMA-SKD S0 (V6-equivalent code, pre-alignment)
!python main.py \
  --data_type cifar100 \
  --data_path /kaggle/working/data \
  --classifier_type ResNet18 \
  --batch_size {_bs} \
  --end_epoch {END_EPOCH} \
  --workers {_wk} \
  --seed 2024 \
  --beta 0.5 \
  --EHSKD \
  {_ddp_flag} \
  --dist_url tcp://127.0.0.1:8081 \
  --experiment_type s0_emaskd_v6equiv


In [ ]:
# Cell 6: Metrics — parse logs and compare baseline vs EMA-SKD
import glob, re
import pandas as pd

def parse_log(path):
    rows = []
    with open(path) as f:
        for line in f:
            if '[val]' not in line:
                continue
            def g(key):
                m = re.search(rf'\[{key} ([^\]]+)\]', line)
                return float(m.group(1)) if m else None
            ep = re.search(r'\[Epoch (\d+)\]', line)
            if not ep:
                continue
            rows.append({
                'epoch':    int(ep.group(1)),
                'top1':     g('val_top1_acc'),
                'top5':     g('val_top5_acc'),
                'val_loss': g('val_loss'),
                'ece':      g('ECE'),
                'aurc':     g('AURC'),
                'eaurc':    g('EAURC'),
            })
    return pd.DataFrame(rows).set_index('epoch')

def find_latest_log(keyword):
    matches = sorted(glob.glob(f'models/*{keyword}*/log/log.txt'))
    if not matches:
        raise FileNotFoundError(f'No log matching: {keyword}')
    return matches[-1]

# Auto-detect: baseline = no EHSKD, ema = EHSKD_True
all_logs = sorted(glob.glob('models/*/log/log.txt'))
base_logs = [p for p in all_logs if 'EHSKD_False' in p]
ema_logs  = [p for p in all_logs if 'EHSKD_True'  in p]

if not base_logs or not ema_logs:
    raise FileNotFoundError(f'Logs not found.\nBaseline: {base_logs}\nEMA: {ema_logs}')

baseline_log = base_logs[-1]
ema_log      = ema_logs[-1]

df_base = parse_log(baseline_log)
df_ema  = parse_log(ema_log)

print('Baseline log:', baseline_log)
print('EMA-SKD log :', ema_log)
print(f'Epochs — Baseline: {len(df_base)}  EMA-SKD: {len(df_ema)}')
print()

last_base = df_base.iloc[-1]
last_ema  = df_ema.iloc[-1]

summary = pd.DataFrame({
    'Metric':   ['Top-1 Acc (%)', 'Top-5 Acc (%)', 'ECE (↓)', 'AURC (↓)', 'EAURC (↓)'],
    'Baseline': [last_base.top1, last_base.top5, last_base.ece, last_base.aurc, last_base.eaurc],
    'EMA-SKD':  [last_ema.top1,  last_ema.top5,  last_ema.ece,  last_ema.aurc,  last_ema.eaurc],
})
summary['Δ'] = summary['EMA-SKD'] - summary['Baseline']
print(summary.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
print()
print('Paper targets — Baseline: 75.55 ± 0.09  |  EMA-SKD: 79.19 ± 0.15')

In [ ]:
# Cell 7: Training curves — Top-1, Val Loss, ECE, AURC per epoch
import glob, re, matplotlib.pyplot as plt, matplotlib.ticker as ticker
import pandas as pd

def _parse_log(path):
    rows = []
    with open(path) as f:
        for line in f:
            if '[val]' not in line: continue
            def g(key):
                m = re.search(rf'\[{key} ([^\]]+)\]', line)
                return float(m.group(1)) if m else None
            ep = re.search(r'\[Epoch (\d+)\]', line)
            if not ep: continue
            rows.append({'epoch': int(ep.group(1)), 'top1': g('val_top1_acc'),
                         'val_loss': g('val_loss'), 'ece': g('ECE'), 'aurc': g('AURC')})
    return pd.DataFrame(rows).set_index('epoch')

all_logs  = sorted(glob.glob('models/*/log/log.txt'))
base_logs = [p for p in all_logs if 'EHSKD_False' in p]
ema_logs  = [p for p in all_logs if 'EHSKD_True'  in p]
if not base_logs or not ema_logs:
    raise FileNotFoundError(f'Logs not found. Baseline={base_logs} EMA={ema_logs}')
_df_base = _parse_log(base_logs[-1])
_df_ema  = _parse_log(ema_logs[-1])

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('EMA-SKD vs Baseline — CIFAR-100 / ResNet18', fontsize=13)
panels = [
    ('top1',     'Top-1 Accuracy (%)', False),
    ('val_loss', 'Val Loss',           True),
    ('ece',      'ECE (↓)',            True),
    ('aurc',     'AURC (↓)',           True),
]
for ax, (col, title, _) in zip(axes.flat, panels):
    ax.plot(_df_base.index, _df_base[col], label='Baseline', marker='o', linewidth=1.5)
    ax.plot(_df_ema.index,  _df_ema[col],  label='EMA-SKD',  marker='s', linewidth=1.5)
    ax.set_title(title); ax.set_xlabel('Epoch')
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/eval_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: /kaggle/working/eval_curves.png')

# Crossover detection
_merged = _df_ema[['top1']].rename(columns={'top1':'ema'}).join(
    _df_base[['top1']].rename(columns={'top1':'base'}), how='inner')
_cross = _merged[_merged['ema'] > _merged['base']]
if _cross.empty:
    print('EMA-SKD did not exceed baseline — need more epochs or check config')
else:
    ep = _cross.index[0]
    print(f'EMA-SKD first exceeds baseline at epoch {ep}'
          f' (EMA {_cross.loc[ep,"ema"]:.3f}% vs Base {_cross.loc[ep,"base"]:.3f}%)')

# Final-epoch delta
print()
for col, title, _ in panels:
    b = _df_base[col].iloc[-1]; e = _df_ema[col].iloc[-1]
    print(f'{title:20s}  Baseline={b:.3f}  EMA-SKD={e:.3f}  Δ={e-b:+.3f}')


In [ ]:
# Cell 8: Resource usage — GPU state + training duration
import glob, re, os, subprocess
import matplotlib.pyplot as plt
from datetime import datetime

# GPU state
gpu_info = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.used,memory.total,utilization.gpu,temperature.gpu',
     '--format=csv,noheader,nounits'],
    capture_output=True, text=True).stdout.strip()
print('=== GPU State (post-training) ===')
for i, line in enumerate(gpu_info.splitlines()):
    p = [x.strip() for x in line.split(',')]
    if len(p) >= 5:
        print(f'GPU {i}: {p[0]}  VRAM: {p[1]}/{p[2]} MiB  Util: {p[3]}%  Temp: {p[4]}°C')
print()

# Training duration from log timestamps
def _duration(log_path):
    with open(log_path) as f:
        lines = f.readlines()
    pat = re.compile(r'\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})')
    times = [pat.search(l) for l in lines]
    times = [t.group(1) for t in times if t]
    if len(times) < 2: return None
    fmt = '%Y-%m-%d %H:%M:%S'
    return (datetime.strptime(times[-1], fmt) - datetime.strptime(times[0], fmt)).total_seconds()

all_logs  = sorted(glob.glob('models/*/log/log.txt'))
base_logs = [p for p in all_logs if 'EHSKD_False' in p]
ema_logs  = [p for p in all_logs if 'EHSKD_True'  in p]

labels, durations, epochs_count = [], [], []
for label, logs in [('Baseline', base_logs), ('EMA-SKD', ema_logs)]:
    if not logs: continue
    path = logs[-1]
    sec  = _duration(path)
    # count epochs from log
    n_epochs = sum(1 for l in open(path) if '[val]' in l and '[Epoch' in l)
    labels.append(label)
    durations.append(sec or 0)
    epochs_count.append(n_epochs)
    size_kb = os.path.getsize(path) / 1024
    avg_sec = (sec / n_epochs) if n_epochs else 0
    print(f'{label:10s}: total={sec:.0f}s  epochs={n_epochs}  avg={avg_sec:.1f}s/epoch  log={size_kb:.0f}KB')

# Bar chart
if labels:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle('Training Resource Usage', fontsize=12)
    axes[0].bar(labels, durations, color=['steelblue', 'darkorange'])
    axes[0].set_ylabel('Total time (s)'); axes[0].set_title('Training Duration')
    for i, v in enumerate(durations):
        axes[0].text(i, v + 10, f'{v:.0f}s', ha='center', fontsize=10)
    avg_per_epoch = [d/e if e else 0 for d, e in zip(durations, epochs_count)]
    axes[1].bar(labels, avg_per_epoch, color=['steelblue', 'darkorange'])
    axes[1].set_ylabel('Seconds / epoch'); axes[1].set_title('Avg Time per Epoch')
    for i, v in enumerate(avg_per_epoch):
        axes[1].text(i, v + 0.5, f'{v:.1f}s', ha='center', fontsize=10)
    plt.tight_layout()
    plt.savefig('/kaggle/working/resource_usage.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved: /kaggle/working/resource_usage.png')
